# State of Data Brazil — Pipeline de Dados (Camada Gold, PySpark)

**Tech Challenge — Indicadores analíticos**

Constrói, a partir da Silver unificada (2023+2024+2025, já limpa e padronizada),
a camada Gold: uma tabela base (`gold_respondentes`) e **7 tabelas agregadas**
("marts"), cada uma desenhada para responder diretamente a uma das perguntas de
negócio do desafio.

| # | Tabela | Pergunta de negócio |
|---|--------|----------------------|
| 1 | `gold_respondentes` | (base — suporta consultas ad hoc) |
| 2 | `gold_panorama_mercado` | Como está estruturado o mercado brasileiro de Dados? |
| 3 | `gold_perfis_profissionais` | Quais perfis profissionais são mais valorizados pelo mercado? |
| 4 | `gold_diversidade` | Qual é o cenário de diversidade de gênero nas carreiras de dados? |
| 5 | `gold_adocao_tecnologias` | Quais tecnologias apresentam maior adoção entre os profissionais? |
| 6 | `gold_adocao_ia` | Qual é o índice de adoção de IA e seu impacto? |
| 7 | `gold_perfil_regional` | Diferenças entre regiões, senioridades ou modelos de trabalho? |
| 8 | `gold_maturidade_e_desafios` | Oportunidades e desafios para empresas investirem em Dados/IA? |

Cada tabela é salva em `data/gold/<nome>/` em Parquet — mesmo layout que sobe
para `s3://<bucket>/gold/<nome>/` e é catalogado via Glue Crawler para consulta
no Athena.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve() / 'src'))
from pipeline_utils import (
    find_project_root, get_spark_session,
    explode_multiselect, descobrir_vocabulario_multiselect, explode_multiselect_por_vocabulario,
)

from pyspark.sql import functions as F

PROJECT_ROOT = find_project_root()
SILVER_DIR = PROJECT_ROOT / 'data' / 'silver'
GOLD_DIR = PROJECT_ROOT / 'data' / 'gold'
GOLD_DIR.mkdir(parents=True, exist_ok=True)

spark = get_spark_session('state-of-data-gold')
print("Raiz do projeto:", PROJECT_ROOT)


c:\Users\fhca02\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Raiz do projeto: c:\Users\fhca02\OneDrive - Church of Jesus Christ\Documents\Fabricio\Pos FIAP\Fase 3\Tech Challenge - Fase 3\state-of-data-pipeline_1


In [2]:
df_silver = spark.read.parquet(str(SILVER_DIR / 'pesquisa_unificada'))
df_silver.cache()
print(f"Silver: {df_silver.count():,} linhas x {len(df_silver.columns)} colunas".replace(',', '.'))


Silver: 14.005 linhas x 83 colunas


## 1. `gold_respondentes` — tabela base

Um subconjunto curado de ~30 campos (dos 80 da Silver) com os atributos mais
usados para fatiar as análises: demografia, formação, situação de trabalho,
cargo/senioridade/salário, satisfação/carreira, adoção de IA e maturidade de
dados na empresa. Grão: **1 linha por respondente por ano** (`token` +
`_ano_pesquisa`). Serve tanto como fonte para os marts abaixo quanto para
consultas ad hoc no Athena que não se encaixem em nenhum mart pré-agregado.


In [3]:
COLUNAS_RESPONDENTES = [
    '_ano_pesquisa', 'token',
    # demografia
    'faixa_idade', 'genero', 'cor_raca_etnia', 'pcd', 'regiao_onde_mora', 'uf_onde_mora',
    # formacao
    'nivel_de_ensino', 'area_de_formacao',
    # trabalho
    'situacao_de_trabalho', 'setor', 'numero_de_funcionarios',
    'cargo_atual', 'nivel', 'faixa_salarial',
    'tempo_de_experiencia_em_dados', 'tempo_de_experiencia_em_ti',
    'atua_como_gestor', 'modelo_de_trabalho_atual', 'modelo_de_trabalho_ideal',
    # satisfacao / carreira
    'satisfeito_atualmente', 'planos_de_mudar_de_emprego_6m', 'participou_de_entrevistas_ultimos_6m',
    # ia
    'ai_generativa_e_llm_e_uma_prioridade', 'usa_chatgpt_ou_copilot_no_trabalho',
    'empresa_esta_conseguindo_ter_bons_resultados_com_llms',
    # maturidade de dados na empresa
    'possui_data_lake', 'possui_data_warehouse',
]

gold_respondentes = df_silver.select(*COLUNAS_RESPONDENTES)
print(f"gold_respondentes: {gold_respondentes.count():,} linhas x {len(gold_respondentes.columns)} colunas".replace(',', '.'))
gold_respondentes.limit(3).toPandas()


gold_respondentes: 14.005 linhas x 29 colunas


c:\Users\fhca02\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,_ano_pesquisa,token,faixa_idade,genero,cor_raca_etnia,pcd,regiao_onde_mora,uf_onde_mora,nivel_de_ensino,area_de_formacao,...,modelo_de_trabalho_atual,modelo_de_trabalho_ideal,satisfeito_atualmente,planos_de_mudar_de_emprego_6m,participou_de_entrevistas_ultimos_6m,ai_generativa_e_llm_e_uma_prioridade,usa_chatgpt_ou_copilot_no_trabalho,empresa_esta_conseguindo_ter_bons_resultados_com_llms,possui_data_lake,possui_data_warehouse
0,2023,001b2d1qtli8t9z7oqgdhj001b2d4i0g,30-34,Masculino,Branca,Não,Sudeste,MG,Doutorado ou Phd,Computação / Engenharia de Software / Sistemas...,...,Modelo híbrido flexível (o funcionário tem lib...,Modelo 100% remoto,Não,Estou em busca de oportunidades dentro ou fora...,"Sim, fiz entrevistas mas não fui aprovado (ou ...",NaN,Não utilizo nenhum tipo de solução de IA Gener...,NaN,NaN,NaN
1,2023,0026aa3fwd78u0026asg7456tfkjg2cs,30-34,Masculino,Branca,Não,Sudeste,ES,Graduação/Bacharelado,Computação / Engenharia de Software / Sistemas...,...,Modelo 100% remoto,Modelo 100% remoto,Sim,"Não estou buscando, mas me considero aberto a ...","Sim, fui aprovado mas decidi não mudar de emprego",NaN,Utilizo soluções pagas de AI Generativa (como ...,NaN,NaN,NaN
2,2023,00r21rb9pusd1b0v7ew00r21rw3dy69w,35-39,Feminino,Amarela,Não,Sudeste,SP,Graduação/Bacharelado,Computação / Engenharia de Software / Sistemas...,...,Modelo híbrido flexível (o funcionário tem lib...,Modelo 100% remoto,Sim,"Não estou buscando, mas me considero aberto a ...",Não participei de entrevistas de emprego/proce...,NaN,Utilizo apenas soluções gratuitas (como por ex...,NaN,NaN,NaN


## 2. `gold_panorama_mercado` — Q1: estrutura do mercado

Grão: ano × tipo de vínculo × setor × porte da empresa × modelo de trabalho.
Contagem de respondentes por combinação — a base para entender como o mercado
de dados está distribuído (tipos de contratação, setores que mais contratam,
porte de empresa predominante, presencial vs. remoto).


In [4]:
gold_panorama_mercado = (
    df_silver
    .groupBy('_ano_pesquisa', 'situacao_de_trabalho', 'setor', 'numero_de_funcionarios', 'modelo_de_trabalho_atual')
    .agg(F.count('*').alias('qtd_respondentes'))
    .orderBy('_ano_pesquisa', F.desc('qtd_respondentes'))
)
print(f"{gold_panorama_mercado.count():,} linhas".replace(',', '.'))
gold_panorama_mercado.show(10, truncate=False)


2.791 linhas
+-------------+--------------------------------------+------------------------------+----------------------+--------------------------------------------------------------------------------------------------------------+----------------+
|_ano_pesquisa|situacao_de_trabalho                  |setor                         |numero_de_funcionarios|modelo_de_trabalho_atual                                                                                      |qtd_respondentes|
+-------------+--------------------------------------+------------------------------+----------------------+--------------------------------------------------------------------------------------------------------------+----------------+
|2023         |Desempregado, buscando recolocação    |NULL                          |NULL                  |NULL                                                                                                          |361             |
|2023         |Empregado (CLT)         

## 3. `gold_perfis_profissionais` — Q2: perfis mais valorizados

Grão: ano × cargo × senioridade × faixa salarial. Contagem de respondentes —
cruzando cargo/senioridade com a distribuição salarial dá o indicador direto
de "quais perfis o mercado mais remunera/demanda".


In [5]:
gold_perfis_profissionais = (
    df_silver
    .groupBy('_ano_pesquisa', 'cargo_atual', 'nivel', 'faixa_salarial')
    .agg(F.count('*').alias('qtd_respondentes'))
    .orderBy('_ano_pesquisa', F.desc('qtd_respondentes'))
)
print(f"{gold_perfis_profissionais.count():,} linhas".replace(',', '.'))
gold_perfis_profissionais.show(10, truncate=False)


1.027 linhas
+-------------+------------------------------------------------+------+--------------------------------+----------------+
|_ano_pesquisa|cargo_atual                                     |nivel |faixa_salarial                  |qtd_respondentes|
+-------------+------------------------------------------------+------+--------------------------------+----------------+
|2023         |NULL                                            |NULL  |NULL                            |540             |
|2023         |NULL                                            |NULL  |de R$ 12.001/mês a R$ 16.000/mês|172             |
|2023         |NULL                                            |NULL  |de R$ 16.001/mês a R$ 20.000/mês|166             |
|2023         |Analista de Dados/Data Analyst                  |Sênior|de R$ 8.001/mês a R$ 12.000/mês |141             |
|2023         |NULL                                            |NULL  |de R$ 8.001/mês a R$ 12.000/mês |126             |
|2023      

## 4. `gold_diversidade` — Q3: diversidade de gênero

Grão: ano × cargo × senioridade × gênero × cor/raça/etnia × PCD. Contagem de
respondentes — permite cruzar cargo/senioridade (proxy de posições de maior
remuneração/liderança) com gênero e outros recortes de diversidade.


In [6]:
gold_diversidade = (
    df_silver
    .groupBy('_ano_pesquisa', 'cargo_atual', 'nivel', 'genero', 'cor_raca_etnia', 'pcd')
    .agg(F.count('*').alias('qtd_respondentes'))
    .orderBy('_ano_pesquisa', F.desc('qtd_respondentes'))
)
print(f"{gold_diversidade.count():,} linhas".replace(',', '.'))
gold_diversidade.show(10, truncate=False)


1.256 linhas
+-------------+------------------------------------------------+------+---------+--------------+---+----------------+
|_ano_pesquisa|cargo_atual                                     |nivel |genero   |cor_raca_etnia|pcd|qtd_respondentes|
+-------------+------------------------------------------------+------+---------+--------------+---+----------------+
|2023         |NULL                                            |NULL  |Masculino|Branca        |Não|700             |
|2023         |NULL                                            |NULL  |Masculino|Parda         |Não|243             |
|2023         |NULL                                            |NULL  |Feminino |Branca        |Não|224             |
|2023         |Engenheiro de Dados/Data Engineer/Data Architect|Sênior|Masculino|Branca        |Não|187             |
|2023         |Analista de Dados/Data Analyst                  |Pleno |Masculino|Branca        |Não|167             |
|2023         |Cientista de Dados/Data Scie

## 5. `gold_adocao_tecnologias` — Q4: adoção de tecnologias

Os campos de tecnologia do dia a dia (`banco_de_dados_dia_a_dia`,
`cloud_dia_a_dia`, `linguagem_de_programacao_dia_a_dia`,
`ferramenta_de_bi_dia_a_dia`) são **multi-select** (várias tecnologias juntas
numa célula, separadas por vírgula) — cada um é "explodido" (uma linha por
tecnologia citada) e depois unidos num único mart com uma coluna `categoria`
identificando de qual pergunta veio.

Grão: ano × categoria × tecnologia. `pct_respondentes_ano` mostra o percentual
do total de respondentes daquele ano que citou aquela tecnologia — como cada
pessoa pode citar várias, o percentual **não soma 100%** dentro de uma
categoria/ano (é normal e esperado para uma pergunta multi-select).


In [7]:
CAMPOS_TECNOLOGIA = [
    ('banco_de_dados_dia_a_dia', 'Banco de Dados'),
    ('cloud_dia_a_dia', 'Cloud'),
    ('linguagem_de_programacao_dia_a_dia', 'Linguagem de Programação'),
    ('ferramenta_de_bi_dia_a_dia', 'Ferramenta de BI'),
]

total_respondentes_ano = (
    df_silver.groupBy('_ano_pesquisa').agg(F.countDistinct('token').alias('total_respondentes_ano'))
)

tecnologias_explodidas = []
for coluna, categoria in CAMPOS_TECNOLOGIA:
    base = df_silver.select('_ano_pesquisa', 'token', coluna).filter(F.col(coluna).isNotNull())
    explodido = explode_multiselect(base, coluna, coluna_saida='tecnologia')
    explodido = explodido.withColumn('categoria', F.lit(categoria))
    tecnologias_explodidas.append(explodido)

df_tecnologias = tecnologias_explodidas[0]
for df_extra in tecnologias_explodidas[1:]:
    df_tecnologias = df_tecnologias.unionByName(df_extra)

gold_adocao_tecnologias = (
    df_tecnologias
    .groupBy('_ano_pesquisa', 'categoria', 'tecnologia')
    .agg(F.countDistinct('token').alias('qtd_respondentes'))
    .join(total_respondentes_ano, on='_ano_pesquisa')
    .withColumn('pct_respondentes_ano', F.round(F.col('qtd_respondentes') / F.col('total_respondentes_ano') * 100, 1))
    .drop('total_respondentes_ano')
    .orderBy('_ano_pesquisa', 'categoria', F.desc('qtd_respondentes'))
)
print(f"{gold_adocao_tecnologias.count():,} linhas".replace(',', '.'))
gold_adocao_tecnologias.filter(F.col('categoria') == 'Linguagem de Programação').show(10, truncate=False)


538 linhas
+-------------+------------------------+-------------------------------------------+----------------+--------------------+
|_ano_pesquisa|categoria               |tecnologia                                 |qtd_respondentes|pct_respondentes_ano|
+-------------+------------------------+-------------------------------------------+----------------+--------------------+
|2023         |Linguagem de Programação|SQL                                        |3156            |59.6                |
|2023         |Linguagem de Programação|Python                                     |2825            |53.4                |
|2023         |Linguagem de Programação|R                                          |401             |7.6                 |
|2023         |Linguagem de Programação|Não utilizo nenhuma das linguagens listadas|305             |5.8                 |
|2023         |Linguagem de Programação|Visual Basic/VBA                           |285             |5.4                 |
|2023

## 6. `gold_adocao_ia` — Q5: adoção de IA e impacto

Três sinais diferentes sobre IA generativa/LLMs, todos possíveis de cruzar com
cargo e setor, são combinados num único mart em formato longo
(`dimensao` identifica qual pergunta originou a linha):

- **`uso_individual_ia`**: `usa_chatgpt_ou_copilot_no_trabalho` — também
  multi-select (uma pessoa pode usar ChatGPT *e* Copilot, por exemplo), mas
  as próprias opções são frases longas que **contêm vírgulas internas** (ex:
  `"...(exemplo: Github Copilot, Amazon CodeWhisperer ou ChatGPT Plus)..."`)
  — um split ingênuo por vírgula quebraria a opção no meio. Por isso usa
  `descobrir_vocabulario_multiselect` (reconstrói as 8 opções atômicas a
  partir das células que contêm só 1 resposta, valida contra as 40 células
  distintas da base) + `explode_multiselect_por_vocabulario`, em vez do
  split simples usado nos demais campos.
- **`prioridade_empresa_ia`**: `ai_generativa_e_llm_e_uma_prioridade` — quão
  prioritário IA Generativa é para a empresa do respondente.
- **`resultado_llms_empresa`**: `empresa_esta_conseguindo_ter_bons_resultados_com_llms`
  — se a empresa está conseguindo gerar resultado de negócio com LLMs (o
  "impacto" da pergunta de negócio).

Grão: ano × dimensão × cargo × setor × valor.


In [8]:
uso_individual_base = df_silver.select('_ano_pesquisa', 'token', 'cargo_atual', 'setor', 'usa_chatgpt_ou_copilot_no_trabalho') \
    .filter(F.col('usa_chatgpt_ou_copilot_no_trabalho').isNotNull())

valores_distintos = [r[0] for r in uso_individual_base.select('usa_chatgpt_ou_copilot_no_trabalho').distinct().collect()]
vocabulario_ia = descobrir_vocabulario_multiselect(valores_distintos)
print(f"{len(vocabulario_ia)} opcoes atomicas descobertas para 'usa_chatgpt_ou_copilot_no_trabalho':")
for opt in vocabulario_ia:
    print(' -', opt[:90] + ('...' if len(opt) > 90 else ''))

uso_individual = explode_multiselect_por_vocabulario(
    uso_individual_base, 'usa_chatgpt_ou_copilot_no_trabalho', vocabulario_ia, coluna_saida='valor'
)
uso_individual = uso_individual.withColumn('dimensao', F.lit('uso_individual_ia'))

prioridade_empresa = df_silver.select('_ano_pesquisa', 'token', 'cargo_atual', 'setor',
                                       F.col('ai_generativa_e_llm_e_uma_prioridade').alias('valor')) \
    .filter(F.col('valor').isNotNull()) \
    .withColumn('dimensao', F.lit('prioridade_empresa_ia'))

resultado_llms = df_silver.select('_ano_pesquisa', 'token', 'cargo_atual', 'setor',
                                   F.col('empresa_esta_conseguindo_ter_bons_resultados_com_llms').alias('valor')) \
    .filter(F.col('valor').isNotNull()) \
    .withColumn('dimensao', F.lit('resultado_llms_empresa'))

df_ia = uso_individual.unionByName(prioridade_empresa).unionByName(resultado_llms)

gold_adocao_ia = (
    df_ia
    .groupBy('_ano_pesquisa', 'dimensao', 'cargo_atual', 'setor', 'valor')
    .agg(F.countDistinct('token').alias('qtd_respondentes'))
    .orderBy('_ano_pesquisa', 'dimensao', F.desc('qtd_respondentes'))
)
print(f"{gold_adocao_ia.count():,} linhas".replace(',', '.'))
gold_adocao_ia.filter(F.col('dimensao') == 'uso_individual_ia').show(8, truncate=False)


8 opcoes atomicas descobertas para 'usa_chatgpt_ou_copilot_no_trabalho':
 - Utilizo soluções pagas de AI Generativa (como por exemplo ChatGPT plus, Anthropic Claude, ...
 - Utilizo soluções no estilo "Copilot" (exemplo: Github Copilot, Amazon CodeWhisperer ou Cha...
 - Utilizo soluções pagas de AI Generativa (como por exemplo ChatGPT plus, Anthropic Claude, ...
 - Utilizo soluções pagas de AI Generativa (como por exemplo ChatGPT plus, MidJourney etc) e ...
 - Utilizo soluções de AI para código (exemplo: Github Copilot, Cursor, Claude Code) para ter...
 - Utilizo soluções pagas de AI Generativa (como por exemplo ChatGPT plus, MidJourney etc) e ...
 - Utilizo apenas soluções gratuitas (como por exemplo o ChatGPT), para me ajudar a ser mais ...
 - Não utilizo nenhum tipo de solução de IA Generativa para melhorar a produtividade no dia a...
2.473 linhas
+-------------+-----------------+------------------------------------------------+------------------------------+-------------------------

## 7. `gold_perfil_regional` — Q6: região, senioridade e modelo de trabalho

Grão: ano × região × senioridade × modelo de trabalho. Contagem de
respondentes — permite comparar, por exemplo, se o modelo 100% remoto é mais
comum fora do Sudeste, ou se a distribuição de senioridade varia por região.


In [9]:
gold_perfil_regional = (
    df_silver
    .groupBy('_ano_pesquisa', 'regiao_onde_mora', 'nivel', 'modelo_de_trabalho_atual')
    .agg(F.count('*').alias('qtd_respondentes'))
    .orderBy('_ano_pesquisa', F.desc('qtd_respondentes'))
)
print(f"{gold_perfil_regional.count():,} linhas".replace(',', '.'))
gold_perfil_regional.show(10, truncate=False)


314 linhas
+-------------+----------------+------+--------------------------------------------------------------------------------------------------------------+----------------+
|_ano_pesquisa|regiao_onde_mora|nivel |modelo_de_trabalho_atual                                                                                      |qtd_respondentes|
+-------------+----------------+------+--------------------------------------------------------------------------------------------------------------+----------------+
|2023         |Sudeste         |Sênior|Modelo 100% remoto                                                                                            |458             |
|2023         |Sudeste         |Pleno |Modelo 100% remoto                                                                                            |372             |
|2023         |Sudeste         |NULL  |NULL                                                                                                          

## 8. `gold_maturidade_e_desafios` — Q7: oportunidades e desafios

Combina, no mesmo formato longo do mart de IA, cinco sinais sobre maturidade
de dados e riscos organizacionais — todos cruzáveis por setor e porte de
empresa:

- **`possui_data_lake`** / **`possui_data_warehouse`**: maturidade de
  infraestrutura de dados (Sim/Não).
- **`desafio_gestor`**: principais desafios relatados por quem gerencia times
  de dados (`desafios_como_gestor`, multi-select → explodido).
- **`motivo_nao_uso_ia`**: motivos para a empresa não usar IA Generativa
  (`motivos_para_nao_usar_ai_generativa_e_llm`, multi-select → explodido) —
  a "oportunidade" para empresas que querem investir em IA é justamente
  endereçar esses motivos.
- **`risco_turnover`**: intenção de troca de emprego nos próximos 6 meses
  (`planos_de_mudar_de_emprego_6m`) — indicador de risco de retenção de
  talento para empresas que investem em times de dados.

Grão: ano × dimensão × setor × porte da empresa × valor.


In [10]:
def preparar_dimensao(coluna, nome_dimensao, multiselect=False):
    base = df_silver.select('_ano_pesquisa', 'token', 'setor', 'numero_de_funcionarios',
                             F.col(coluna).alias(coluna)) \
        .filter(F.col(coluna).isNotNull())
    if multiselect:
        base = explode_multiselect(base, coluna, coluna_saida='valor')
    else:
        base = base.withColumnRenamed(coluna, 'valor')
    return base.withColumn('dimensao', F.lit(nome_dimensao))

dimensoes = [
    preparar_dimensao('possui_data_lake', 'possui_data_lake'),
    preparar_dimensao('possui_data_warehouse', 'possui_data_warehouse'),
    preparar_dimensao('desafios_como_gestor', 'desafio_gestor', multiselect=True),
    preparar_dimensao('motivos_para_nao_usar_ai_generativa_e_llm', 'motivo_nao_uso_ia', multiselect=True),
    preparar_dimensao('planos_de_mudar_de_emprego_6m', 'risco_turnover'),
]

df_maturidade = dimensoes[0]
for d in dimensoes[1:]:
    df_maturidade = df_maturidade.unionByName(d)

gold_maturidade_e_desafios = (
    df_maturidade
    .groupBy('_ano_pesquisa', 'dimensao', 'setor', 'numero_de_funcionarios', 'valor')
    .agg(F.countDistinct('token').alias('qtd_respondentes'))
    .orderBy('_ano_pesquisa', 'dimensao', F.desc('qtd_respondentes'))
)
print(f"{gold_maturidade_e_desafios.count():,} linhas".replace(',', '.'))
gold_maturidade_e_desafios.filter(F.col('dimensao') == 'desafio_gestor').show(8, truncate=False)


7.266 linhas
+-------------+--------------+------------------+----------------------+-----------------------------------------------------------------------------------------+----------------+
|_ano_pesquisa|dimensao      |setor             |numero_de_funcionarios|valor                                                                                    |qtd_respondentes|
+-------------+--------------+------------------+----------------------+-----------------------------------------------------------------------------------------+----------------+
|2023         |desafio_gestor|Finanças ou Bancos|Acima de 3.000        |Gerenciar a expectativa das áreas de negócio em relação as entregas das equipes de dados.|29              |
|2023         |desafio_gestor|Varejo            |Acima de 3.000        |Gerenciar a expectativa das áreas de negócio em relação as entregas das equipes de dados.|29              |
|2023         |desafio_gestor|Finanças ou Bancos|Acima de 3.000        |Organizar as in

## 9. Checagem de qualidade das tabelas Gold

In [11]:
tabelas_gold = {
    'gold_respondentes': gold_respondentes,
    'gold_panorama_mercado': gold_panorama_mercado,
    'gold_perfis_profissionais': gold_perfis_profissionais,
    'gold_diversidade': gold_diversidade,
    'gold_adocao_tecnologias': gold_adocao_tecnologias,
    'gold_adocao_ia': gold_adocao_ia,
    'gold_perfil_regional': gold_perfil_regional,
    'gold_maturidade_e_desafios': gold_maturidade_e_desafios,
}

for nome, df in tabelas_gold.items():
    print(f"{nome:32s} {df.count():>8,} linhas x {len(df.columns)} colunas".replace(',', '.'))


gold_respondentes                  14.005 linhas x 29 colunas
gold_panorama_mercado               2.791 linhas x 6 colunas
gold_perfis_profissionais           1.027 linhas x 5 colunas
gold_diversidade                    1.256 linhas x 7 colunas
gold_adocao_tecnologias               538 linhas x 5 colunas
gold_adocao_ia                      2.473 linhas x 6 colunas
gold_perfil_regional                  314 linhas x 5 colunas
gold_maturidade_e_desafios          7.266 linhas x 6 colunas


In [12]:
# gold_respondentes nao pode ter tokens duplicados dentro do mesmo ano (mesma checagem da Silver)
dup = (
    gold_respondentes.groupBy('_ano_pesquisa', 'token').count()
    .filter(F.col('count') > 1)
    .count()
)
print(f"Tokens duplicados em gold_respondentes: {dup}")

# nas tabelas agregadas, qtd_respondentes nunca deve ser <= 0
for nome, df in tabelas_gold.items():
    if nome == 'gold_respondentes':
        continue
    invalidas = df.filter(F.col('qtd_respondentes') <= 0).count()
    status = 'OK' if invalidas == 0 else f'ATENCAO: {invalidas} linhas com qtd_respondentes <= 0'
    print(f"{nome}: {status}")


Tokens duplicados em gold_respondentes: 3
gold_panorama_mercado: OK
gold_perfis_profissionais: OK
gold_diversidade: OK
gold_adocao_tecnologias: OK
gold_adocao_ia: OK
gold_perfil_regional: OK
gold_maturidade_e_desafios: OK


## 10. Persistência da camada Gold

In [13]:
for nome, df in tabelas_gold.items():
    out_path = GOLD_DIR / nome
    df.coalesce(1).write.mode('overwrite').parquet(str(out_path))
    print(f"Salvo: {out_path}")


Salvo: c:\Users\fhca02\OneDrive - Church of Jesus Christ\Documents\Fabricio\Pos FIAP\Fase 3\Tech Challenge - Fase 3\state-of-data-pipeline_1\data\gold\gold_respondentes
Salvo: c:\Users\fhca02\OneDrive - Church of Jesus Christ\Documents\Fabricio\Pos FIAP\Fase 3\Tech Challenge - Fase 3\state-of-data-pipeline_1\data\gold\gold_panorama_mercado
Salvo: c:\Users\fhca02\OneDrive - Church of Jesus Christ\Documents\Fabricio\Pos FIAP\Fase 3\Tech Challenge - Fase 3\state-of-data-pipeline_1\data\gold\gold_perfis_profissionais
Salvo: c:\Users\fhca02\OneDrive - Church of Jesus Christ\Documents\Fabricio\Pos FIAP\Fase 3\Tech Challenge - Fase 3\state-of-data-pipeline_1\data\gold\gold_diversidade
Salvo: c:\Users\fhca02\OneDrive - Church of Jesus Christ\Documents\Fabricio\Pos FIAP\Fase 3\Tech Challenge - Fase 3\state-of-data-pipeline_1\data\gold\gold_adocao_tecnologias
Salvo: c:\Users\fhca02\OneDrive - Church of Jesus Christ\Documents\Fabricio\Pos FIAP\Fase 3\Tech Challenge - Fase 3\state-of-data-pipeline

In [14]:
for nome, df in tabelas_gold.items():
    out_path = GOLD_DIR / nome
    df_check = spark.read.parquet(str(out_path))
    status = 'OK' if (df_check.count(), len(df_check.columns)) == (df.count(), len(df.columns)) else 'DIVERGENTE'
    print(f"{nome:32s} shape lido {df_check.count():>8,}x{len(df_check.columns)} | {status}".replace(',', '.'))


gold_respondentes                shape lido   14.005x29 | OK
gold_panorama_mercado            shape lido    2.791x6 | OK
gold_perfis_profissionais        shape lido    1.027x5 | OK
gold_diversidade                 shape lido    1.256x7 | OK
gold_adocao_tecnologias          shape lido      538x5 | OK
gold_adocao_ia                   shape lido    2.473x6 | OK
gold_perfil_regional             shape lido      314x5 | OK
gold_maturidade_e_desafios       shape lido    7.266x6 | OK


## 11. Próximos passos

- **Athena / SQL**: com as 8 tabelas em `s3://<bucket>/gold/<nome>/` e
  catalogadas via Glue Crawler, as 7 perguntas de negócio viram consultas SQL
  diretas sobre os marts correspondentes (sem precisar reprocessar a base
  toda a cada pergunta).
- **DataViz**: cada mart já está no grão certo para virar 1-2 gráficos direto
  (ex: `gold_adocao_tecnologias` → gráfico de barras por categoria/tecnologia).
- **AWS**: mesmo código roda como Glue Job — troca `get_spark_session` local
  por `GlueContext` e os paths locais por `s3://<bucket>/...`.
